# Tier 5 Pipeline — Harmonizing the Data of Your Data
## NCEMS Kaggle Competition | Full Production Notebook

**Pipeline stages:**
1. Setup & config
2. Load training data (papers + gold SDRFs + GPT extracts)
3. Build correction map from `Training_GPT_Extract` vs. gold
4. Build TF-IDF retriever for RAG (few-shot context)
5. OLS API ontology resolver (EFO / MONDO / UBERON / PSI-MS / NCBITaxon)
6. LLM extraction with GPT-4o (structured JSON + retry)
7. Post-processing: ontology normalization, correction map, NT= formatter
8. SDRF validation with `sdrf-pipelines`
9. Local scoring with `src/Scoring.py`
10. Generate `submission.csv`


In [1]:
# Install required packages
import subprocess, sys

pkgs = [
    "torch",
    "transformers>=4.40",
    "accelerate",
    "scikit-learn",
    "pandas",
    "numpy",
    "rapidfuzz",
    "requests",
    "tqdm",
    "sdrf-pipelines",
    "sentence-transformers",
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("✅ All packages installed")


✅ All packages installed


## 1. Configuration

In [20]:
import os, json, re, difflib, time, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR        = Path("..").resolve()
DATA_DIR        = BASE_DIR / "data"
TRAIN_TEXT_DIR  = DATA_DIR / "TrainingPubText"
TRAIN_SDRF_DIR  = DATA_DIR / "TrainingSDRFs"
TRAIN_GPT_DIR   = DATA_DIR / "Training_GPT_Extract"
TEST_TEXT_DIR   = DATA_DIR / "TestPubText"
BASELINE_PROMPT = DATA_DIR / "BaselinePrompt.txt"
SAMPLE_SUB      = DATA_DIR / "SampleSubmission.csv"
SCORING_PY      = BASE_DIR / "src" / "Scoring.py"

OUTPUT_DIR      = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Local LLM ─────────────────────────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS  = 512    # flat extraction dict — doesn't need more
MAX_PAPER_CHARS = 2_000  # ~500 tokens — fits comfortably in the 1.5B context

# ── Scoring ────────────────────────────────────────────────────────────────────
CLUSTER_THRESHOLD = 0.80

print("Config loaded. BASE_DIR =", BASE_DIR)
print(f"Local LLM: {MODEL_ID}  MAX_PAPER_CHARS={MAX_PAPER_CHARS}  MAX_NEW_TOKENS={MAX_NEW_TOKENS}")


Config loaded. BASE_DIR = C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data
Local LLM: Qwen/Qwen2.5-1.5B-Instruct  MAX_PAPER_CHARS=2000  MAX_NEW_TOKENS=512


## 2. Load Template & Training Data

In [5]:
# Load the exact submission template (defines all 81 columns)
template_df = pd.read_csv(SAMPLE_SUB)
ALL_COLS     = list(template_df.columns)
META_COLS    = [c for c in ALL_COLS if c not in ("ID", "PXD", "Raw Data File", "Usage")]

print(f"Template: {len(ALL_COLS)} columns, {len(META_COLS)} metadata columns")
print("First 10 meta cols:", META_COLS[:10])


Template: 81 columns, 77 metadata columns
First 10 meta cols: ['Characteristics[Age]', 'Characteristics[AlkylationReagent]', 'Characteristics[AnatomicSiteTumor]', 'Characteristics[AncestryCategory]', 'Characteristics[BMI]', 'Characteristics[Bait]', 'Characteristics[BiologicalReplicate]', 'Characteristics[CellLine]', 'Characteristics[CellPart]', 'Characteristics[CellType]']


In [7]:
def load_json_text(path: Path) -> dict:
    """Load a publication JSON file. Returns dict with at least 'full_text'."""
    with open(path) as f:
        data = json.load(f)
    if "full_text" not in data:
        for key in ("body", "text", "content", "abstract"):
            if key in data:
                data["full_text"] = data[key]
                break
        else:
            data["full_text"] = " ".join(str(v) for v in data.values() if isinstance(v, str))
    return data


def load_gpt_metadata_txt(path: Path) -> dict:
    """
    Parse a GPT metadata .txt file with 'Key: Value' lines into a dict.
    Multi-value keys (same key appearing more than once) become lists.
    """
    result: Dict[str, object] = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or ":" not in line:
                continue
            key, _, value = line.partition(":")
            key   = key.strip()
            value = value.strip()
            if not key:
                continue
            # Map bare key names to SDRF column names where possible
            if key in result:
                existing = result[key]
                if isinstance(existing, list):
                    existing.append(value)
                else:
                    result[key] = [existing, value]
            else:
                result[key] = value
    return result


# Load all training papers
train_papers: Dict[str, dict] = {}
for fp in sorted(TRAIN_TEXT_DIR.glob("*.json")):
    pxd = fp.stem
    train_papers[pxd] = load_json_text(fp)
print(f"Loaded {len(train_papers)} training papers")

# Load gold-standard SDRFs (TSV format, stem = PXD*_cleaned.sdrf)
train_sdrfs: Dict[str, pd.DataFrame] = {}
for fp in sorted(TRAIN_SDRF_DIR.glob("*.tsv")):
    pxd = fp.stem.split("_")[0]   # e.g. "PXD000070_cleaned.sdrf" -> "PXD000070"
    train_sdrfs[pxd] = pd.read_csv(fp, sep="\t")
print(f"Loaded {len(train_sdrfs)} training SDRFs")

# Load GPT extracts — pick the most recent model subfolder
gpt_root = TRAIN_GPT_DIR / "GPT_Extract"
gpt_subdirs = sorted(gpt_root.glob("*/")) if gpt_root.exists() else []
gpt_dir = gpt_subdirs[-1] if gpt_subdirs else None   # last = alphabetically latest

train_gpt: Dict[str, dict] = {}
if gpt_dir:
    for fp in sorted(gpt_dir.glob("*_Metadata.txt")):
        pxd = fp.stem.replace("_Metadata", "")
        train_gpt[pxd] = load_gpt_metadata_txt(fp)
    print(f"Loaded {len(train_gpt)} GPT extract files (from {gpt_dir.name})")
else:
    print("⚠️  No GPT extract folder found")

# Load test papers
test_papers: Dict[str, dict] = {}
for fp in sorted(TEST_TEXT_DIR.glob("*.json")):
    pxd = fp.stem
    test_papers[pxd] = load_json_text(fp)
print(f"Loaded {len(test_papers)} test papers")


Loaded 104 training papers
Loaded 103 training SDRFs
Loaded 103 GPT extract files (from o4-mini-2025-04-16)
Loaded 16 test papers


## 3. Build Correction Map from GPT Extracts vs. Gold SDRFs

In [8]:
def build_correction_map(
    train_sdrfs: Dict[str, pd.DataFrame],
    train_gpt:   Dict[str, dict],
    threshold:   float = 0.82,
) -> Dict[str, str]:
    """
    For each column, compare GPT-extracted values to gold-standard values.
    When a GPT value is close to (but not equal to) a gold value, record the mapping.
    Returns: {gpt_value -> gold_value}
    """
    correction: Dict[str, str] = {}

    common_pxds = set(train_sdrfs) & set(train_gpt)
    for pxd in common_pxds:
        gold_df  = train_sdrfs[pxd]
        gpt_data = train_gpt[pxd]

        for col in META_COLS:
            # Gold values for this column
            if col not in gold_df.columns:
                continue
            gold_vals = set(
                str(v).strip() for v in gold_df[col].dropna()
                if str(v).strip() not in ("", "nan", "not available", "Not Applicable")
            )
            if not gold_vals:
                continue

            # GPT values: may be list or single string
            raw_gpt = gpt_data.get(col, [])
            if isinstance(raw_gpt, str):
                raw_gpt = [raw_gpt]
            elif not isinstance(raw_gpt, list):
                raw_gpt = [str(raw_gpt)]

            gpt_vals = {str(v).strip() for v in raw_gpt if str(v).strip()}

            for gv in gpt_vals:
                if gv in gold_vals:
                    continue  # already correct
                # Find closest gold value
                best_gold, best_sim = None, 0.0
                for gold_v in gold_vals:
                    sim = difflib.SequenceMatcher(None, gv.lower(), gold_v.lower()).ratio()
                    if sim > best_sim:
                        best_sim, best_gold = sim, gold_v
                if best_sim >= threshold and best_gold and gv != best_gold:
                    correction[gv] = best_gold

    print(f"Correction map built: {len(correction)} entries")
    return correction

correction_map = build_correction_map(train_sdrfs, train_gpt)

# Show sample corrections
for k, v in list(correction_map.items())[:15]:
    print(f"  {k!r:50s} -> {v!r}")


Correction map built: 0 entries


## 4. TF-IDF Retriever for RAG Few-Shot Context

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_pxd_list  = sorted(train_papers.keys())
train_text_list = [train_papers[p]["full_text"][:MAX_PAPER_CHARS] for p in train_pxd_list]

vectorizer   = TfidfVectorizer(max_features=15_000, sublinear_tf=True, stop_words="english")
train_matrix = vectorizer.fit_transform(train_text_list)

def retrieve_similar(query_text: str, top_k: int = 3) -> List[Tuple[str, float]]:
    """Return [(pxd, similarity_score), ...] for the top-k closest training papers."""
    q_vec = vectorizer.transform([query_text[:MAX_PAPER_CHARS]])
    sims  = cosine_similarity(q_vec, train_matrix)[0]
    top_i = np.argsort(sims)[::-1][:top_k]
    return [(train_pxd_list[i], float(sims[i])) for i in top_i]

# Quick sanity check
example_pxd  = list(test_papers.keys())[0]
example_hits = retrieve_similar(test_papers[example_pxd]["full_text"])
print(f"Test PXD {example_pxd} → closest training papers:")
for pxd, sim in example_hits:
    print(f"  {pxd}  (sim={sim:.3f})")


Test PXD PubText → closest training papers:
  PubText  (sim=0.000)
  PXD021874_PubText  (sim=0.000)
  PXD003531_PubText  (sim=0.000)


## 5. OLS4 API Ontology Resolver

In [10]:
import requests
from functools import lru_cache

OLS_BASE = "https://www.ebi.ac.uk/ols4/api/search"

# Column → (ontology, format_string)
# format_string uses {label} and {obo_id} placeholders
COLUMN_ONTOLOGY: Dict[str, Tuple[str, str]] = {
    "Characteristics[Organism]":         ("ncbitaxon", "NT={label};AC={obo_id}"),
    "Characteristics[Disease]":          ("mondo",     "NT={label};AC={obo_id}"),
    "Characteristics[OrganismPart]":     ("uberon",    "NT={label};AC={obo_id}"),
    "Characteristics[CellType]":         ("cl",        "NT={label};AC={obo_id}"),
    "Characteristics[CellLine]":         ("efo",       "NT={label};AC={obo_id}"),
    "Characteristics[DevelopmentalStage]":("efo",      "NT={label};AC={obo_id}"),
    "Characteristics[Sex]":              ("pato",      "NT={label};AC={obo_id}"),
    "Characteristics[MaterialType]":     ("efo",       "NT={label};AC={obo_id}"),
    "Characteristics[CleavageAgent]":    ("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[AlkylationReagent]":("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[ReductionReagent]": ("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[Label]":            ("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[Modification]":     ("ms",        "NT={label};MT=Variable;AC={obo_id}"),
    "Characteristics[Modification].1":   ("ms",        "NT={label};MT=Variable;AC={obo_id}"),
    "Characteristics[Modification].2":   ("ms",        "NT={label};MT=Variable;AC={obo_id}"),
    "Characteristics[Modification].3":   ("ms",        "NT={label};MT=Fixed;AC={obo_id}"),
    "Characteristics[Modification].4":   ("ms",        "NT={label};MT=Fixed;AC={obo_id}"),
    "Comment[Instrument]":               ("ms",        "NT={label};AC={obo_id}"),
    "Comment[FragmentationMethod]":      ("ms",        "NT={label};AC={obo_id}"),
    "Comment[MS2MassAnalyzer]":          ("ms",        "NT={label};AC={obo_id}"),
    "Comment[IonizationType]":           ("ms",        "NT={label};AC={obo_id}"),
    "Comment[AcquisitionMethod]":        ("ms",        "NT={label};AC={obo_id}"),
    "Comment[EnrichmentMethod]":         ("ms",        "NT={label};AC={obo_id}"),
    "Comment[FractionationMethod]":      ("ms",        "NT={label};AC={obo_id}"),
    "Comment[Separation]":               ("ms",        "NT={label};AC={obo_id}"),
}

_ols_cache: Dict[Tuple[str,str], Optional[Tuple[str,str]]] = {}

def ols_lookup(term: str, ontology: str) -> Optional[Tuple[str, str]]:
    """
    Query OLS4 for `term` in `ontology`.
    Returns (label, obo_id) of the best hit, or None.
    """
    key = (term.lower().strip(), ontology)
    if key in _ols_cache:
        return _ols_cache[key]

    try:
        resp = requests.get(
            OLS_BASE,
            params={"q": term, "ontology": ontology, "rows": 5, "exact": "false"},
            timeout=10,
        )
        resp.raise_for_status()
        docs = resp.json().get("response", {}).get("docs", [])
        if docs:
            best = docs[0]
            label  = best.get("label", term)
            obo_id = best.get("obo_id", "")
            result = (label, obo_id)
        else:
            result = None
    except Exception:
        result = None

    _ols_cache[key] = result
    return result


def resolve_to_sdrf_term(raw_value: str, col: str) -> str:
    """
    Given a raw extracted value and the SDRF column name, return the
    ontology-formatted string (NT=...;AC=...) if a mapping exists,
    else return the raw value cleaned up.
    """
    v = raw_value.strip()
    if not v or v.lower() in ("not available", "not applicable", "na", "n/a", ""):
        return "not available"

    # Already formatted
    if v.startswith("NT="):
        return v

    if col in COLUMN_ONTOLOGY:
        ontology, fmt = COLUMN_ONTOLOGY[col]
        hit = ols_lookup(v, ontology)
        if hit:
            label, obo_id = hit
            obo_id_clean = obo_id.replace(":", "_") if obo_id else ""
            # Use the AC in colon format for SDRF (e.g. MS:1001251)
            ac = obo_id if ":" in obo_id else obo_id_clean
            return fmt.format(label=label, obo_id=ac)

    return v


# Test the resolver
test_cases = [
    ("Homo sapiens",    "Characteristics[Organism]"),
    ("liver cancer",    "Characteristics[Disease]"),
    ("blood plasma",    "Characteristics[OrganismPart]"),
    ("trypsin",         "Characteristics[CleavageAgent]"),
    ("Q Exactive HF",   "Comment[Instrument]"),
    ("HCD",             "Comment[FragmentationMethod]"),
    ("label free",      "Characteristics[Label]"),
    ("Oxidation",       "Characteristics[Modification]"),
]
for raw, col in test_cases:
    result = resolve_to_sdrf_term(raw, col)
    print(f"  {raw:25s} -> {result}")


  Homo sapiens              -> NT=Homo sapiens;AC=NCBITaxon:9606
  liver cancer              -> NT=liver cancer;AC=MONDO:0002691
  blood plasma              -> NT=blood plasma;AC=UBERON:0001969
  trypsin                   -> NT=Trypsin;AC=MS:1001251
  Q Exactive HF             -> NT=Q Exactive HF;AC=MS:1002523
  HCD                       -> NT=beam-type collision-induced dissociation;AC=MS:1000422
  label free                -> label free
  Oxidation                 -> Oxidation


## 6. LLM Extraction with GPT-4o

In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ── Load model ────────────────────────────────────────────────────────────────
_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_ID} on {_device} ...")

_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if _device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
_model.eval()

_pipe = pipeline(
    "text-generation",
    model=_model,
    tokenizer=_tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    return_full_text=False,
)
print("✅ Model ready.")

# Load baseline prompt (used only to log; not injected into every call)
if BASELINE_PROMPT.exists():
    with open(BASELINE_PROMPT) as f:
        BASELINE_PROMPT_TEXT = f.read()
    print(f"BaselinePrompt.txt loaded ({len(BASELINE_PROMPT_TEXT)} chars) — used as reference only for 1.5B model")
else:
    BASELINE_PROMPT_TEXT = ""
    print("⚠️  BaselinePrompt.txt not found")

# ── Compact prompt designed for a 1.5B model ─────────────────────────────────
# The full BaselinePrompt is too long for 1.5B.
# Instead: short system + flat-dict output (one key per SDRF field, string values).
# Post-processing then maps the flat dict to per-file rows using RAW_FILES.

_KEY_FIELDS = [
    "Organism", "OrganismPart", "Disease", "CellType", "CellLine",
    "Sex", "Age", "Strain", "DevelopmentalStage", "MaterialType",
    "CleavageAgent", "ReductionReagent", "AlkylationReagent", "Label",
    "Modification",
    "Instrument", "AcquisitionMethod", "FragmentationMethod", "IonizationType",
    "MS2MassAnalyzer", "EnrichmentMethod", "FractionationMethod",
    "GradientTime", "FlowRateChromatogram",
    "PrecursorMassTolerance", "FragmentMassTolerance", "NumberOfMissedCleavages",
    "NumberOfFractions", "NumberOfBiologicalReplicates", "NumberOfTechnicalReplicates",
    "Treatment", "Compound", "Temperature", "Time",
    "BiologicalReplicate", "TechnicalReplicate",
    "TumorStage", "TumorGrade", "TumorSite",
    "RawDataFiles",
]

_FIELD_LIST_STR = ", ".join(f'"{k}"' for k in _KEY_FIELDS)

SYSTEM_PROMPT = (
    "You are a proteomics metadata extractor. "
    "Read the manuscript text and return ONLY a JSON object with these exact keys:\n"
    f"{_FIELD_LIST_STR}\n\n"
    "Rules:\n"
    "- Each value is a string (or list of strings for Modification and RawDataFiles).\n"
    "- Use exact phrases from the text. Use null if not mentioned.\n"
    "- RawDataFiles: list of .raw/.mzML/.d file names mentioned in the paper.\n"
    "- Output ONLY JSON. No markdown, no explanation."
)

# Map compact keys → SDRF column names
_KEY_TO_SDRF: Dict[str, str] = {
    "Organism":               "Characteristics[Organism]",
    "OrganismPart":           "Characteristics[OrganismPart]",
    "Disease":                "Characteristics[Disease]",
    "CellType":               "Characteristics[CellType]",
    "CellLine":               "Characteristics[CellLine]",
    "Sex":                    "Characteristics[Sex]",
    "Age":                    "Characteristics[Age]",
    "Strain":                 "Characteristics[Strain]",
    "DevelopmentalStage":     "Characteristics[DevelopmentalStage]",
    "MaterialType":           "Characteristics[MaterialType]",
    "CleavageAgent":          "Characteristics[CleavageAgent]",
    "ReductionReagent":       "Characteristics[ReductionReagent]",
    "AlkylationReagent":      "Characteristics[AlkylationReagent]",
    "Label":                  "Characteristics[Label]",
    "Modification":           "Characteristics[Modification]",
    "Instrument":             "Comment[Instrument]",
    "AcquisitionMethod":      "Comment[AcquisitionMethod]",
    "FragmentationMethod":    "Comment[FragmentationMethod]",
    "IonizationType":         "Comment[IonizationType]",
    "MS2MassAnalyzer":        "Comment[MS2MassAnalyzer]",
    "EnrichmentMethod":       "Comment[EnrichmentMethod]",
    "FractionationMethod":    "Comment[FractionationMethod]",
    "GradientTime":           "Comment[GradientTime]",
    "FlowRateChromatogram":   "Comment[FlowRateChromatogram]",
    "PrecursorMassTolerance": "Comment[PrecursorMassTolerance]",
    "FragmentMassTolerance":  "Comment[FragmentMassTolerance]",
    "NumberOfMissedCleavages":"Comment[NumberOfMissedCleavages]",
    "NumberOfFractions":      "Comment[NumberOfFractions]",
    "NumberOfBiologicalReplicates": "Characteristics[NumberOfBiologicalReplicates]",
    "NumberOfTechnicalReplicates":  "Characteristics[NumberOfTechnicalReplicates]",
    "Treatment":              "Characteristics[Treatment]",
    "Compound":               "Characteristics[Compound]",
    "Temperature":            "Characteristics[Temperature]",
    "Time":                   "Characteristics[Time]",
    "BiologicalReplicate":    "Characteristics[BiologicalReplicate]",
    "TechnicalReplicate":     "Characteristics[TechnicalReplicate]",
    "TumorStage":             "Characteristics[TumorStage]",
    "TumorGrade":             "Characteristics[TumorGrade]",
    "TumorSite":              "Characteristics[TumorSite]",
    "RawDataFiles":           "Raw Data Files",  # handled specially
}


def extract_sdrf_llm(
    paper_text:   str,
    pxd:          str,
    raw_files:    List[str],
    similar_pxds: List[Tuple[str, float]],
    retries:      int = 2,
) -> Optional[dict]:
    """
    Call the local LLM with a compact prompt.
    Returns a flat dict with short keys (e.g. "Organism") mapped to string/list values,
    plus a "Raw Data Files" list.  Per-file expansion is done in postprocess_extraction.
    """
    user_msg = f"MANUSCRIPT_TEXT:\n{paper_text[:MAX_PAPER_CHARS]}"
    if raw_files:
        user_msg += f"\n\nRAW_FILES: {json.dumps(raw_files[:30])}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    for attempt in range(retries):
        try:
            raw_out = _pipe(prompt)[0]["generated_text"]
            raw_out = re.sub(r"^```(?:json)?\s*", "", raw_out.strip(), flags=re.IGNORECASE)
            raw_out = re.sub(r"\s*```$", "", raw_out.strip())
            match = re.search(r"\{.*\}", raw_out, re.DOTALL)
            parsed = json.loads(match.group() if match else raw_out)
            if isinstance(parsed, dict):
                # Remap to SDRF column names
                remapped: dict = {}
                for k, v in parsed.items():
                    sdrf_col = _KEY_TO_SDRF.get(k, k)
                    remapped[sdrf_col] = v
                # Ensure Raw Data Files
                if "Raw Data Files" not in remapped:
                    remapped["Raw Data Files"] = raw_files or []
                return remapped
        except json.JSONDecodeError as e:
            print(f"  [Attempt {attempt+1}] JSON parse error for {pxd}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"  [Attempt {attempt+1}] Error for {pxd}: {e}")
            time.sleep(2)
    return None


Loading Qwen/Qwen2.5-1.5B-Instruct on cuda ...


Loading weights: 100%|██████████| 338/338 [00:09<00:00, 37.17it/s]


✅ Model ready.
BaselinePrompt.txt loaded (13217 chars) — used as reference only for 1.5B model


## 7. Post-Processing: Ontology Normalization + Correction Map

In [17]:
def apply_correction_map(value: str, correction_map: Dict[str, str]) -> str:
    """Apply direct and case-insensitive corrections from the correction map."""
    if value in correction_map:
        return correction_map[value]
    lower = value.lower()
    for k, v in correction_map.items():
        if k.lower() == lower:
            return v
    return value


def _pick_span(spans) -> str:
    """
    From a list of verbatim spans (new prompt format) or a plain value (old format),
    return a single string to use as the SDRF cell value.
    """
    if isinstance(spans, list):
        # Filter empty / null entries and join multiple spans with " | "
        clean = [str(s).strip() for s in spans if str(s).strip() not in ("", "None", "null")]
        if not clean:
            return "not available"
        return " | ".join(clean)
    if spans is None:
        return "not available"
    return str(spans).strip() or "not available"


def postprocess_extraction(
    extracted: dict,
    correction_map: Dict[str, str],
    pxd: str,
    template_cols: List[str],
) -> List[Dict[str, str]]:
    """
    Convert raw LLM output (new per-file format OR legacy flat dict) into a
    list of SDRF rows aligned with ALL_COLS.

    New format:  {filename: {sdrf_key: [span, ...], ...}, ...}
    Legacy flat: {"Raw Data Files": [...], sdrf_col: value, ...}
    """

    # ── Detect format ─────────────────────────────────────────────────────────
    is_per_file = (
        isinstance(extracted, dict)
        and extracted
        and all(isinstance(v, dict) for v in extracted.values())
        and not any(k in extracted for k in ("Raw Data Files", "Characteristics[Organism]"))
    )

    # Handle legacy flat dict (or __flat__ wrapper from fallback)
    if "__flat__" in extracted:
        extracted = extracted["__flat__"]
        is_per_file = False

    if not is_per_file:
        # ── Legacy path ───────────────────────────────────────────────────────
        raw_files = extracted.get("Raw Data Files", [])
        if not isinstance(raw_files, list) or not raw_files:
            raw_files = ["unknown.raw"]

        mod_vals: List[str] = []
        for key in ("Characteristics[Modification]",):
            val = extracted.get(key, [])
            if isinstance(val, list):
                mod_vals.extend(str(v).strip() for v in val if str(v).strip())
            elif isinstance(val, str) and val.strip():
                mod_vals.append(val.strip())
        mod_vals = list(dict.fromkeys(mod_vals))

        rows = []
        for raw_file in raw_files:
            row = {"ID": str(len(rows) + 1), "PXD": pxd,
                   "Raw Data File": str(raw_file).strip(), "Usage": "Raw Data File"}
            for col in template_cols:
                if col in ("ID", "PXD", "Raw Data File", "Usage"):
                    continue
                if col.startswith("Characteristics[Modification]"):
                    suffix = col.replace("Characteristics[Modification]", "")
                    idx = 0 if suffix == "" else int(suffix.lstrip("."))
                    raw_val = mod_vals[idx] if idx < len(mod_vals) else "not available"
                else:
                    raw_val = _pick_span(extracted.get(col, "not available"))
                if not raw_val or raw_val.lower() in ("none", "null", "nan", ""):
                    raw_val = "not available"
                raw_val = apply_correction_map(raw_val, correction_map)
                row[col] = resolve_to_sdrf_term(raw_val, col)
            rows.append(row)
        return rows

    # ── New per-file path ─────────────────────────────────────────────────────
    rows = []
    for file_key, file_meta in extracted.items():
        # Resolve the actual filename (prefer Raw Data File key if present)
        raw_file = _pick_span(file_meta.get("Raw Data File", file_key))
        if not raw_file or raw_file == "not available":
            raw_file = file_key

        row = {"ID": str(len(rows) + 1), "PXD": pxd,
               "Raw Data File": raw_file, "Usage": "Raw Data File"}

        # Collect Modification spans for numbered columns
        mod_vals = []
        for k, v in file_meta.items():
            if k.startswith("Characteristics[Modification]"):
                spans = v if isinstance(v, list) else [v]
                mod_vals.extend(str(s).strip() for s in spans if str(s).strip())
        mod_vals = list(dict.fromkeys(mod_vals))

        for col in template_cols:
            if col in ("ID", "PXD", "Raw Data File", "Usage"):
                continue
            if col.startswith("Characteristics[Modification]"):
                suffix = col.replace("Characteristics[Modification]", "")
                idx = 0 if suffix == "" else int(suffix.lstrip("."))
                raw_val = mod_vals[idx] if idx < len(mod_vals) else "not available"
            else:
                raw_val = _pick_span(file_meta.get(col, "not available"))

            if not raw_val or raw_val.lower() in ("none", "null", "nan", ""):
                raw_val = "not available"
            raw_val = apply_correction_map(raw_val, correction_map)
            row[col] = resolve_to_sdrf_term(raw_val, col)

        rows.append(row)
    return rows if rows else [{"ID": "1", "PXD": pxd,
                               "Raw Data File": "unknown.raw", "Usage": "Raw Data File"}]


# Sanity test on a training paper (doesn't call the LLM)
_test_pxd  = list(train_sdrfs.keys())[0]
_gold_df   = train_sdrfs[_test_pxd]
# Simulate new-format output using the first two gold rows
_sim_extracted = {}
for _, _row in _gold_df.head(2).iterrows():
    _fname = str(_row.get("Raw Data File", "sample.raw")).strip()
    _fd = {"Raw Data File": [_fname]}
    for _c in META_COLS:
        if _c in _gold_df.columns:
            _v = str(_row.get(_c, "")).strip()
            if _v and _v.lower() not in ("nan", "not available", ""):
                _fd[_c] = [_v]
    _sim_extracted[_fname] = _fd

_test_rows = postprocess_extraction(_sim_extracted, correction_map, _test_pxd, ALL_COLS)
print(f"Sanity test on {_test_pxd}: {len(_test_rows)} rows")
print(pd.DataFrame(_test_rows)[["PXD", "Raw Data File",
    "Characteristics[Organism]", "Comment[Instrument]",
    "Characteristics[Disease]"]].head(3).to_string())


Sanity test on PXD000070: 1 rows
         PXD Raw Data File Characteristics[Organism] Comment[Instrument] Characteristics[Disease]
0  PXD000070    sample.raw             not available       not available            not available


## 8. Local Scoring Utility

In [14]:
import importlib.util, sys as _sys

def load_scoring_module(scoring_py: Path):
    """Dynamically load src/Scoring.py so we can call score() locally."""
    spec   = importlib.util.spec_from_file_location("Scoring", scoring_py)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

if SCORING_PY.exists():
    scoring_module = load_scoring_module(SCORING_PY)
    print("✅ Scoring module loaded from", SCORING_PY)
else:
    scoring_module = None
    print("⚠️  src/Scoring.py not found — local scoring disabled")


def local_score(pred_df: pd.DataFrame, gold_df: pd.DataFrame) -> Tuple[pd.DataFrame, float]:
    """Score predictions against gold SDRF using the competition metric."""
    if scoring_module is None:
        raise RuntimeError("Scoring module not loaded")
    eval_df, score = scoring_module.score(gold_df.copy(), pred_df.copy(), "ID")
    return eval_df, score


def combine_sdrfs(sdrfs: Dict[str, pd.DataFrame], template_cols: List[str]) -> pd.DataFrame:
    """Combine per-PXD SDRF dataframes into one aligned CSV."""
    rows = []
    global_id = 1
    for pxd, df in sdrfs.items():
        for _, row in df.iterrows():
            r = {c: "not available" for c in template_cols}
            r["PXD"] = pxd
            for c in df.columns:
                if c in template_cols:
                    r[c] = row[c]
            r["ID"]    = str(global_id)
            r["Usage"] = "Raw Data File"
            rows.append(r)
            global_id += 1
    return pd.DataFrame(rows)[template_cols]


# Validate training set with GPT extracts (before running the actual LLM)
print("\nRunning training-set validation with GPT extracts (proxy score)...")
if scoring_module:
    sample_pxds  = list(set(train_gpt.keys()) & set(train_sdrfs.keys()))[:5]
    pred_rows    = []
    for pxd in sample_pxds:
        pred_rows.extend(postprocess_extraction(train_gpt[pxd], correction_map, pxd, ALL_COLS))
    pred_df = pd.DataFrame(pred_rows)
    pred_df["ID"] = range(1, len(pred_df) + 1)
    pred_df["ID"] = pred_df["ID"].astype(str)

    gold_rows = []
    for pxd in sample_pxds:
        df = train_sdrfs[pxd].copy()
        df["PXD"] = pxd
        gold_rows.append(df)
    gold_df = pd.concat(gold_rows, ignore_index=True)
    if "ID" not in gold_df.columns:
        gold_df["ID"] = range(1, len(gold_df) + 1)
    gold_df["ID"] = gold_df["ID"].astype(str)

    for c in ALL_COLS:
        if c not in pred_df.columns: pred_df[c] = "not available"
        if c not in gold_df.columns: gold_df[c] = "not available"

    try:
        eval_df, proxy_f1 = local_score(pred_df[ALL_COLS], gold_df[ALL_COLS])
        print(f"Proxy F1 on {len(sample_pxds)} training papers: {proxy_f1:.4f}")
        print("\nPer-annotation breakdown (bottom 10 — areas to improve):")
        print(eval_df.groupby("AnnotationType")["f1"].mean().sort_values().head(10).to_string())
    except Exception as e:
        print(f"Scoring error: {e}")


✅ Scoring module loaded from C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\src\Scoring.py

Running training-set validation with GPT extracts (proxy score)...
Processing PXD=PXD001725, column=PXD, unique values (pre-harmonization): ['PXD001725']
Processing PXD=PXD001725, column=Raw Data File, unique values (pre-harmonization): ['not available']
Processing PXD=PXD001725, column=Characteristics[Age], unique values (pre-harmonization): ['not available']
Processing PXD=PXD001725, column=Characteristics[AlkylationReagent], unique values (pre-harmonization): ['not available']
Processing PXD=PXD001725, column=Characteristics[AnatomicSiteTumor], unique values (pre-harmonization): ['not available']
Processing PXD=PXD001725, column=Characteristics[AncestryCategory], unique values (pre-harmonization): ['not available']
Processing PXD=PXD001725, column=Characteristics[BMI], unique values (pre-harmonization): ['not available']
Processing PXD=PXD001725, column=Characteris

## 9. SDRF Validation with sdrf-pipelines

In [15]:
def validate_sdrf_row(row: dict) -> List[str]:
    """
    Lightweight validation checks on a single SDRF row.
    Returns list of warning strings (empty = OK).
    """
    warnings_list = []

    # 1. Organism should have NT= format
    org = row.get("Characteristics[Organism]", "")
    if org and org not in ("not available",) and not org.startswith("NT="):
        warnings_list.append(f"Organism missing NT= format: {org!r}")

    # 2. Instrument should have NT= format
    inst = row.get("Comment[Instrument]", "")
    if inst and inst not in ("not available",) and not inst.startswith("NT="):
        warnings_list.append(f"Instrument missing NT= format: {inst!r}")

    # 3. Label should be present for quantitative experiments
    label = row.get("Characteristics[Label]", "not available")
    if not label or label == "not available":
        warnings_list.append("Label is not available — check if this is expected")

    # 4. Raw Data File must not be 'unknown'
    rdf = row.get("Raw Data File", "")
    if not rdf or rdf in ("unknown.raw", ""):
        warnings_list.append("Raw Data File is unknown — cannot determine file count")

    # 5. Biological replicate should be numeric or 'not available'
    br = row.get("Characteristics[BiologicalReplicate]", "not available")
    if br not in ("not available", "Not Applicable") and not str(br).isdigit():
        warnings_list.append(f"BiologicalReplicate is non-numeric: {br!r}")

    return warnings_list


def validate_submission_df(df: pd.DataFrame) -> pd.DataFrame:
    """Run validation on all rows. Returns a DataFrame of issues."""
    issues = []
    for i, row in df.iterrows():
        row_warnings = validate_sdrf_row(row.to_dict())
        for w in row_warnings:
            issues.append({"row": i, "PXD": row.get("PXD", "?"), "warning": w})
    if not issues:
        print("✅ No validation issues found")
        return pd.DataFrame()
    warn_df = pd.DataFrame(issues)
    print(f"⚠️  {len(warn_df)} validation warnings")
    print(warn_df.groupby("warning").size().sort_values(ascending=False).head(10))
    return warn_df


## 10. Main Test Extraction Loop

In [22]:
all_pred_rows: List[dict] = []
extraction_log: List[dict] = []

print(f"Extracting SDRF for {len(test_papers)} test papers...\n")

for pxd, paper_data in tqdm(test_papers.items(), desc="Extracting"):
    paper_text = paper_data.get("full_text", "")

    # Collect raw filenames from the paper's JSON (if present)
    raw_files = paper_data.get("raw_files", paper_data.get("files", []))
    if not isinstance(raw_files, list):
        raw_files = []

    # Retrieve similar training papers for few-shot context
    similar = retrieve_similar(paper_text, top_k=3)

    # Call the local LLM
    extracted = extract_sdrf_llm(paper_text, pxd, raw_files, similar)

    if extracted is None:
        print(f"  ❌ Extraction failed for {pxd} — using fallback (not available)")
        extracted = {}

    # Post-process: correction map + OLS ontology resolution
    rows = postprocess_extraction(extracted, correction_map, pxd, ALL_COLS)

    log_entry = {
        "pxd":         pxd,
        "n_rows":      len(rows),
        "n_files":     len(rows),
        "top_similar": similar[0][0] if similar else "none",
        "top_sim":     similar[0][1] if similar else 0.0,
        "organism":    rows[0].get("Characteristics[Organism]", "?") if rows else "?",
        "instrument":  rows[0].get("Comment[Instrument]", "?") if rows else "?",
    }
    extraction_log.append(log_entry)
    all_pred_rows.extend(rows)

print(f"\nTotal rows generated: {len(all_pred_rows)}")
log_df = pd.DataFrame(extraction_log)
print(log_df[["pxd", "n_rows", "top_similar", "top_sim", "organism", "instrument"]].to_string())


Extracting SDRF for 16 test papers...



Extracting:   6%|▋         | 1/16 [11:18<2:49:33, 678.22s/it]


KeyboardInterrupt: 

## 11. Assemble, Validate & Score Submission

In [ ]:
# Assign global IDs
submission_df = pd.DataFrame(all_pred_rows)
submission_df["ID"] = range(1, len(submission_df) + 1)
submission_df["ID"] = submission_df["ID"].astype(str)

# Ensure all columns present and in correct order
for col in ALL_COLS:
    if col not in submission_df.columns:
        submission_df[col] = "not available"
submission_df = submission_df[ALL_COLS]

print(f"Submission shape: {submission_df.shape}")
print(submission_df[["ID", "PXD", "Raw Data File",
                      "Characteristics[Organism]", "Comment[Instrument]",
                      "Characteristics[Disease]"]].head(10).to_string())

# ── Validation ─────────────────────────────────────────────────────────────────
warn_df = validate_submission_df(submission_df)
if not warn_df.empty:
    warn_df.to_csv(OUTPUT_DIR / "validation_warnings.csv", index=False)
    print("Warnings saved to outputs/validation_warnings.csv")


In [ ]:
# ── Optional: cross-validate on a held-out training split ─────────────────────
# (Run this cell BEFORE the test extraction to tune your pipeline)

# Comment out or skip this cell if you've already done the test extraction
CROSS_VAL_PXDS = list(train_papers.keys())[:10]  # use first 10 training papers

print(f"Running cross-validation on {len(CROSS_VAL_PXDS)} training papers...")

cv_pred_rows = []
for pxd in tqdm(CROSS_VAL_PXDS, desc="CV Extract"):
    paper_text = train_papers[pxd].get("full_text", "")
    similar    = retrieve_similar(paper_text, top_k=3)
    # IMPORTANT: exclude the current paper from few-shot (avoid data leakage)
    similar    = [(p, s) for p, s in similar if p != pxd][:2]
    raw_files  = train_papers[pxd].get("raw_files", train_papers[pxd].get("files", []))
    if not isinstance(raw_files, list):
        raw_files = []
    extracted  = extract_sdrf_llm(paper_text, pxd, raw_files, similar)
    if extracted is None:
        extracted = {}
    rows = postprocess_extraction(extracted, correction_map, pxd, ALL_COLS)
    cv_pred_rows.extend(rows)

cv_pred_df = pd.DataFrame(cv_pred_rows)
cv_pred_df["ID"] = range(1, len(cv_pred_df) + 1)
cv_pred_df["ID"] = cv_pred_df["ID"].astype(str)
for c in ALL_COLS:
    if c not in cv_pred_df.columns:
        cv_pred_df[c] = "not available"

# Build gold for same PXDs
cv_gold_rows = []
for pxd in CROSS_VAL_PXDS:
    df = train_sdrfs[pxd].copy()
    df["PXD"] = pxd
    cv_gold_rows.append(df)
cv_gold_df = pd.concat(cv_gold_rows, ignore_index=True)
if "ID" not in cv_gold_df.columns:
    cv_gold_df["ID"] = range(1, len(cv_gold_df) + 1)
cv_gold_df["ID"] = cv_gold_df["ID"].astype(str)
for c in ALL_COLS:
    if c not in cv_gold_df.columns:
        cv_gold_df[c] = "not available"

if scoring_module:
    cv_eval_df, cv_f1 = local_score(cv_pred_df[ALL_COLS], cv_gold_df[ALL_COLS])
    print(f"\n✅ Cross-validation F1: {cv_f1:.4f}")
    print("\nPer-column F1 breakdown:")
    col_f1 = cv_eval_df.groupby("AnnotationType")["f1"].mean().sort_values(ascending=False)
    print(col_f1.to_string())

    cv_eval_df.to_csv(OUTPUT_DIR / "cv_detailed_metrics.csv", index=False)
    print("\nDetailed CV metrics saved to outputs/cv_detailed_metrics.csv")

## 12. Save Final Submission

In [ ]:
sub_path = OUTPUT_DIR / "submission.csv"
submission_df.to_csv(sub_path, index=False)
print(f"✅ Submission saved: {sub_path}")
print(f"   Shape: {submission_df.shape}")

# ── Verify against template ────────────────────────────────────────────────────
template_cols_set  = set(ALL_COLS)
submission_cols_set = set(submission_df.columns)
missing = template_cols_set - submission_cols_set
extra   = submission_cols_set - template_cols_set

if missing: print(f"❌ MISSING columns: {missing}")
if extra:   print(f"⚠️  EXTRA columns (will be ignored): {extra}")
if not missing and not extra:
    print("✅ Column structure matches SampleSubmission.csv exactly")

# Verify no fully-empty rows
null_rows = submission_df.drop(columns=["ID","PXD","Raw Data File","Usage"]).apply(
    lambda row: (row == "not available").all(), axis=1
).sum()
print(f"Rows with ALL metadata = 'not available': {null_rows}/{len(submission_df)}")

print("\nValue distribution for key columns:")
for col in ["Characteristics[Organism]", "Comment[Instrument]", "Characteristics[Disease]"]:
    vc = submission_df[col].value_counts().head(5)
    print(f"\n{col}:")
    print(vc.to_string())


## 13. Diagnostics & Error Analysis

In [ ]:
# Show OLS cache stats
print(f"OLS API cache: {len(_ols_cache)} entries ({sum(v is not None for v in _ols_cache.values())} hits)")

# Show correction map usage
applied_corrections = 0
for _, row in submission_df.iterrows():
    for col in META_COLS:
        val = row.get(col, "")
        if val in correction_map.values():
            applied_corrections += 1
print(f"Correction map entries applied across submission: ~{applied_corrections}")

# Columns with highest 'not available' rate
na_rates = {}
for col in META_COLS:
    na_rate = (submission_df[col] == "not available").mean()
    na_rates[col] = na_rate
na_series = pd.Series(na_rates).sort_values(ascending=False)
print("\nTop 15 columns by 'not available' rate (extraction gaps):")
print(na_series.head(15).to_string())

# Save OLS cache for future runs
import pickle
ols_cache_path = OUTPUT_DIR / "ols_cache.pkl"
with open(ols_cache_path, "wb") as f:
    pickle.dump(_ols_cache, f)
print(f"\nOLS cache saved to {ols_cache_path} (load in future runs to save API calls)")


## (Optional) Load Saved OLS Cache on Subsequent Runs

In [ ]:
import pickle
_ols_cache_path = OUTPUT_DIR / "ols_cache.pkl"
if _ols_cache_path.exists():
    with open(_ols_cache_path, "rb") as f:
        _ols_cache.update(pickle.load(f))
    print(f"Loaded {len(_ols_cache)} OLS cache entries from disk")
else:
    print("No OLS cache file found — starting fresh")


## 14. Package ZIP for Kaggle Submission

In [ ]:
import zipfile, shutil

zip_path = OUTPUT_DIR / "kaggle_submission.zip"
pipeline_path = OUTPUT_DIR / "pipeline.py"

# Write a minimal pipeline.py for documentation
pipeline_src = '''#!/usr/bin/env python3
"""
Tier 5 SDRF Extraction Pipeline
Competition: Harmonizing the Data of Your Data (NCEMS Kaggle)
Approach: GPT-4o + TF-IDF RAG + OLS4 ontology resolution + correction map
"""
# Full implementation in notebook: tier5_sdrf_pipeline.ipynb
# Run the notebook cells in order to regenerate submission.csv
'''
with open(pipeline_path, "w") as f:
    f.write(pipeline_src)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(sub_path,      "submission.csv")
    zf.write(pipeline_path, "pipeline.py")

print(f"✅ Kaggle submission ZIP saved: {zip_path}")
print(f"   Contents: submission.csv + pipeline.py")
print("\nNext steps:")
print("  1. Go to: https://www.kaggle.com/competitions/harmonizing-the-data-of-your-data")
print("  2. Click 'Make a Submission'")
print("  3. Upload kaggle_submission.zip")
